[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch07-frameworks/01-thue-dieu-phoi.ipynb)

# Thuế điều phối: khi phụ trội lớn hơn việc thật

Chương 7 nói framework là một trình biên dịch, và bài toán đầu tiên nó phải giải là thực thi:
phép toán chạy khi nào và chạy thế nào. Ở chế độ thực thi tức thời (eager execution), mỗi phép
đi qua bộ điều phối (dispatch) của framework trước khi kernel được nạp lên thiết bị. Khoản phụ
trội đó nhỏ với một phép, nhưng cộng dồn qua hàng nghìn phép của một lượt truyền xuôi.

Sổ tay này đo khoản phụ trội ấy trên chính CPU bạn đang chạy, rồi dùng mô hình của sách để thấy
vì sao một bước huấn luyện nhỏ trên GPU lại dành gần hết thời gian cho việc điều phối.

**Bạn sẽ làm:**
1. leo cái thang trừu tượng trên một phép nhân ma trận, từ vòng lặp Python tới một lời gọi duy nhất;
2. tính thuế điều phối (dispatch tax) theo mô hình minh hoạ của sách;
3. đo phụ trội điều phối mỗi lần gọi trên máy này, bằng cách chia cùng một lượng việc thành n lần gọi;
4. áp định luật phụ trội điều phối cho hai kịch bản của sách;
5. dựng lại bảng phân tích một bước huấn luyện của chương, rồi đếm và đo chính bước đó trên CPU.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
from collections import Counter

import pandas as pd
import torch.nn.functional as F
from torch.utils._python_dispatch import TorchDispatchMode

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # bốn màu cố định cho các chuỗi số liệu


def thoi_gian(viec, lan=5):
    """Thời gian nhỏ nhất của `lan` lần chạy (giây): ít bị nhiễu bởi việc khác trên máy nhất."""
    viec()
    tot = float("inf")
    for _ in range(lan):
        t0 = time.perf_counter()
        viec()
        tot = min(tot, time.perf_counter() - t0)
    return tot


print("sẵn sàng")

## 1. Cái thang trừu tượng, đo trên một phép nhân ma trận

Năm 1979, viết một phép nhân ma trận chạy nhanh bằng Fortran đòi hiểu dòng cache, cách xếp thanh
ghi và đơn vị vector. Đến năm 2016, một dòng Python đã nạp được một hiện thực đã tối ưu. Bốn bậc
dưới đây làm cùng một phép tính $C = AB$ với $A, B$ cỡ $64 \times 64$, và chỉ khác nhau ở chỗ ranh
giới giữa Python và thư viện nằm đâu:

* **vòng lặp Python**: mỗi phép nhân-cộng là vài lệnh của bộ thông dịch;
* **một lời gọi cho mỗi phần tử**: mỗi phần tử của $C$ là một lời gọi `torch.dot`;
* **một lời gọi cho mỗi hàng**: Python lo vòng lặp ngoài, thư viện lo một hàng (khuôn hình vector
  hoá của NumPy: ngôn ngữ chậm lo logic, ngôn ngữ nhanh lo vòng lặp);
* **một lời gọi**: `A @ B` giao cả phép tính cho thư viện BLAS.

**Dự đoán:** bậc nhanh nhất nhanh hơn bậc chậm nhất bao nhiêu lần? Mười lần, trăm lần, hay hơn nữa?

In [ ]:
n = 64
dat_hat_giong()
A, B = torch.randn(n, n), torch.randn(n, n)
A_ds, B_ds = A.tolist(), B.tolist()            # danh sách Python thuần, cho bậc thứ nhất
hang_A = list(A.unbind(0))                      # 64 hàng của A
cot_B = list(B.t().contiguous().unbind(0))      # 64 cột của B, mỗi cột liền nhau trong bộ nhớ


def vong_lap_python():
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        Ai, Ci = A_ds[i], C[i]
        for k in range(n):
            a, Bk = Ai[k], B_ds[k]
            for j in range(n):
                Ci[j] += a * Bk[j]
    return torch.tensor(C)


def moi_phan_tu():
    return torch.stack([torch.stack([torch.dot(h, c) for c in cot_B]) for h in hang_A])


def moi_hang():
    return torch.stack([h @ B for h in hang_A])


def mot_loi_goi():
    return A @ B


bac = {  # tên bậc: (hàm, số lời gọi torch)
    "vòng lặp Python": (vong_lap_python, 1),
    "một lời gọi mỗi phần tử": (moi_phan_tu, n * n + n + 1),
    "một lời gọi mỗi hàng": (moi_hang, n + 1),
    "một lời gọi": (mot_loi_goi, 1),
}
FLOP = 2 * n**3
ket_qua = {}
for ten, (ham, so_goi) in bac.items():
    assert torch.allclose(ham(), A @ B, atol=1e-4)   # bốn bậc, một đáp số
    ket_qua[ten] = thoi_gian(ham, lan=3)
    do_tren_may(f"{ten} ({so_goi:,} lời gọi torch)", ket_qua[ten] * 1e3, "ms")
do_tren_may("chậm nhất / nhanh nhất", max(ket_qua.values()) / min(ket_qua.values()), "lần")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ten_bac = list(ket_qua)[::-1]
ax.barh(ten_bac, [ket_qua[t] * 1e3 for t in ten_bac], color=MAU[0], height=0.55)
for k, t in enumerate(ten_bac):
    ax.annotate(f"{FLOP / ket_qua[t] / 1e9:.2g} GFLOP/s", (ket_qua[t] * 1e3, k), xytext=(5, 0),
                textcoords="offset points", va="center", fontsize=8)
ax.set_xscale("log")
ax.set_xlim(right=max(ket_qua.values()) * 1e3 * 30)
ax.set_xlabel("thời gian một phép nhân 64×64 (ms, thang log)")
ax.set_title("Cùng 524,288 FLOP, bốn ranh giới Python–thư viện (đo trên máy này)")
plt.tight_layout()
plt.show()

Cả bốn bậc làm đúng $2 \cdot 64^3 = 524,288$ FLOP và cho cùng một ma trận. Khác nhau chỉ ở số lần
chuyển giao giữa Python và thư viện, và ở chỗ thư viện có được thấy cả phép tính một lúc hay không.
Bậc cuối thấy cả phép tính, nên nó chọn được cách chia khối theo cache và dùng đơn vị vector.
Hai bậc giữa thì gần như chỉ trả tiền cho các lần gọi: mỗi lời gọi làm rất ít việc, nên thời gian
của chúng đi theo số lời gọi nhiều hơn là theo số phép tính. Phần còn lại của sổ tay đo chính khoản
đó.

## 2. Thuế điều phối theo mô hình của sách

Napkin Math 1.1 của chương tách phụ trội mỗi phép thành hai phần: điều phối từ Python khoảng 10 µs
và nạp kernel lên GPU khoảng 5 µs. Mô hình minh hoạ của chương gộp lại thành một khoản cố định
$t_{dp}$ cho mỗi phép. Thuế điều phối là phần thời gian nằm ở khoản đó:

$$\text{thuế} = \frac{t_{dp}}{t_{dp} + t_{kernel}}$$

In [ ]:
T_DP = sach(15, nguon="15 µs")   # µs mỗi phép, mô hình minh hoạ của sách
theo_sach("điều phối Python + nạp kernel (µs)",
          sach(10, trich="Python dispatch cost of ~10 μs", tol=0.5) + sach(5, trich="kernel launch cost of ~5 μs", tol=0.5),
          sach=15, nguon="15 µs")


def thue(t_kernel_us, t_dp=T_DP):
    """Phần thời gian nằm ở điều phối, khi kernel chạy t_kernel_us micro giây."""
    return t_dp / (t_dp + t_kernel_us)


relu_nho = sach(1, nguon="1 µs → 94 phần trăm")      # kernel ReLU trên một vector ngắn
gemm_lon = sach(100, nguon="100 µs → 13 phần trăm")  # kernel của một phép nhân ma trận lớn
theo_sach("thuế điều phối, ReLU nhỏ (%)", 100 * thue(relu_nho), sach=94, nguon="1 µs → 94 phần trăm")
theo_sach("thuế điều phối, GEMM lớn (%)", 100 * thue(gemm_lon), sach=13, nguon="100 µs → 13 phần trăm")

t_k = np.logspace(-1, 4, 200)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.semilogx(t_k, 100 * thue(t_k), color=MAU[0], lw=2)
for tk, lech in ((relu_nho, (12, -4)), (gemm_lon, (10, 8))):
    ax.plot(tk, 100 * thue(tk), "o", color=MAU[1], ms=8)
    ax.annotate(f"kernel {tk:g} µs: {100 * thue(tk):.0f} %", (tk, 100 * thue(tk)), xytext=lech,
                textcoords="offset points", fontsize=9)
ax.axvline(T_DP, color="0.6", ls="--", lw=1)
ax.annotate("kernel dài bằng phụ trội: 50 %", (T_DP, 50), xytext=(8, 0), textcoords="offset points",
            fontsize=8, color="0.35")
ax.set_ylim(0, 105)
ax.set_xlabel("thời gian kernel chạy trên thiết bị (µs, thang log)")
ax.set_ylabel("thuế điều phối (%)")
ax.set_title("Cùng một khoản 15 µs, khác kích thước phép toán")
plt.tight_layout()
plt.show()

Cùng một khoản phụ trội, chỉ khác kích thước phép toán, mà tỉ lệ đi từ 94 xuống 13 phần trăm. Ở
phía trái, GPU dành gần hết thời gian chờ lệnh kế tiếp. Ở phía phải, hệ thống chuyển sang nghẽn ở
tính toán (compute-bound) và phụ trội gần như không còn đáng kể.

## 3. Đo phụ trội điều phối trên máy này

Sách nói phụ trội thật đổi theo framework, backend và phần cứng. Vậy trên máy này nó là bao nhiêu?

Cách đo (phỏng theo phần A của phòng thí nghiệm đi kèm chương): giữ nguyên lượng việc, là cộng hai
vector $2^{20}$ phần tử FP32, và chỉ đổi số lần gọi $n$. Ta chia mỗi vector thành $n$ mảnh liền nhau
rồi gọi `torch.add` cho từng mảnh, ghi vào một vùng đã cấp phát sẵn. Số phép cộng và số byte đọc
ghi không đổi, chỉ số lần điều phối đổi. Nếu mô hình của sách đúng thì

$$T(n) \approx n \cdot t_{dp} + W,$$

một đường thẳng theo $n$: độ dốc là phụ trội mỗi lần gọi, tung độ gốc $W$ là phần việc thật.

**Dự đoán:** chia thành bao nhiêu mảnh thì phụ trội bằng đúng phần việc thật? Mỗi mảnh khi đó còn
bao nhiêu phần tử?

In [ ]:
M = 2**20
a, b, c = torch.randn(M), torch.randn(M), torch.empty(M)
cac_n = [1, 4, 16, 64, 256, 1024, 4096, 16384]
do_n = []
for k in cac_n:
    manh_a, manh_b, manh_c = a.view(k, -1).unbind(0), b.view(k, -1).unbind(0), c.view(k, -1).unbind(0)

    def cong_tung_manh():
        for x_, y_, z_ in zip(manh_a, manh_b, manh_c):
            torch.add(x_, y_, out=z_)

    do_n.append(thoi_gian(cong_tung_manh))
    assert torch.equal(c, a + b)   # cùng một kết quả, dù chia bao nhiêu mảnh

# Khớp T = n·t_dp + W theo sai số tương đối: các điểm trải từ 0.1 ms tới hàng chục ms, nên khớp
# thường sẽ chỉ nghe theo vài điểm lớn nhất.
ns, Ts = np.array(cac_n, float), np.array(do_n)
t_dp_may, W_may = np.linalg.lstsq(np.stack([ns, np.ones_like(ns)], 1) / Ts[:, None], np.ones_like(Ts), rcond=None)[0]
n_giao = W_may / t_dp_may
do_tren_may("phụ trội mỗi lần gọi torch.add, t_dp", t_dp_may * 1e6, "µs")
do_tren_may("phần việc thật W (cộng 2^20 phần tử)", W_may * 1e3, "ms")
do_tren_may("số lần gọi để phụ trội bằng việc thật", n_giao)
do_tren_may("khi đó mỗi mảnh còn", M / n_giao, "phần tử")
for k in cac_n:
    # công thức thuế của phần 2, với kernel thay bằng phần việc của một mảnh, W/n
    do_tren_may(f"n = {k:>5}: thuế điều phối theo khớp", 100 * k * t_dp_may / (k * t_dp_may + W_may), "%")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(cac_n, np.array(do_n) * 1e3, "o", color=MAU[0], ms=8, label="đo được")
n_min = np.logspace(0, np.log10(cac_n[-1]), 100)
ax.loglog(n_min, (n_min * t_dp_may + W_may) * 1e3, color=MAU[0], lw=1.5, label="khớp n·t_dp + W")
ax.loglog(n_min, n_min * t_dp_may * 1e3, color=MAU[1], ls="--", lw=1.2, label="chỉ phụ trội, n·t_dp")
ax.axhline(W_may * 1e3, color=MAU[2], ls=":", lw=1.5, label="chỉ việc thật, W")
ax.plot(n_giao, 2 * W_may * 1e3, "s", color=MAU[1], ms=8)
ax.annotate(f"phụ trội = việc thật\nn ≈ {n_giao:,.0f}", (n_giao, 2 * W_may * 1e3), xytext=(-110, 18),
            textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_ylim(bottom=W_may * 1e3 / 4)
ax.set_xlabel("số lần gọi torch.add cho cùng 2^20 phép cộng")
ax.set_ylabel("thời gian (ms)")
ax.set_title("Cùng một lượng việc, chỉ đổi số lần điều phối (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Bên trái điểm giao, thời gian gần như phẳng: phần việc thật chi phối, và thêm vài lần gọi không
đáng kể. Bên phải, đường cong bám theo đường phụ trội: chia việc nhỏ hơn nữa chỉ thêm thời gian
điều phối, dù số phép cộng không đổi. Đó là trần hiệu năng của thực thi tức thời mà chương mô tả,
đo được ở ngay đây.

Trên CPU không có bước nạp kernel lên GPU, nên con số $t_{dp}$ ở trên chỉ là phần phía máy chủ:
Python, bộ điều phối của PyTorch và việc kiểm tra kiểu. Trên GPU còn phải cộng thêm khoản nạp kernel.

## 4. Định luật phụ trội điều phối

Phương trình 4 của chương đặt khoản phụ trội cạnh thời gian phần cứng:

$$\text{tỉ số phụ trội} = \frac{N_{ops} \cdot t_{dp}}{T_{compute} + T_{memory}}$$

Khi tỉ số lớn hơn 1, mô hình bị phụ trội chi phối. Mẫu số dùng phép cộng dồn không chồng lấp; khi
việc chạy thật có chồng lấp thì thay bằng thời gian phần cứng đo được. Napkin Math 1.2 của chương
áp phương trình cho hai kịch bản ở hai đầu của phổ.

In [ ]:
def ti_so_phu_troi(n_ops, t_dp, t_hw):
    """Phương trình 4 của chương, với t_hw = T_compute + T_memory (cùng đơn vị với t_dp)."""
    return n_ops * t_dp / t_hw


# Kịch bản một: một MLP nhỏ, sáu phép ma trận nhỏ và theo phần tử.
n_ops_mlp = sach(6, trich="Compute: 6 ops across small matrix/element-wise operations")
t_hw_mlp = sach(2.6, trich="2.6 μs (mostly memory latency)")
t_sw_mlp = n_ops_mlp * T_DP
theo_sach("phụ trội phần mềm của MLP nhỏ (µs)", t_sw_mlp, sach=90, trich="6 ops \\(\\times\\) 15 μs/op = 90 μs")
theo_sach("tỉ số phụ trội", ti_so_phu_troi(n_ops_mlp, T_DP, t_hw_mlp), sach=34.6,
          trich="Ratio: 90 μs/2.6 μs ≈ 34.6.", tol=0.05)
theo_sach("phần thời gian nằm ở điều phối (%)", 100 * t_sw_mlp / (t_sw_mlp + t_hw_mlp), sach=97,
          trich="The system spends 97 percent of time")
theo_sach("tăng tốc tối đa nếu xoá hết phụ trội", (t_sw_mlp + t_hw_mlp) / t_hw_mlp, sach=35.6,
          trich="upper-bound speedup of 35.6×")

# Kịch bản hai: một tầng GPT-3, những phép nhân ma trận khổng lồ.
t_hw_gpt3 = sach(100000, trich="100 ms = 100000 μs")
t_sw_gpt3 = sach(50, trich="\\(T_{\\text{sw}} \\approx 50 \\, \\mu s\\)", tol=0.5)
theo_sach("tỉ số phụ trội của một tầng GPT-3", t_sw_gpt3 / t_hw_gpt3, sach=0.0005,
          trich="Ratio: 50 μs/100000 μs ≈ 0.0005.", tol=0.00005)

In [ ]:
t_hw = np.logspace(-1, 6, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(t_hw, t_sw_mlp / t_hw, color=MAU[0], lw=2, label=f"phụ trội {t_sw_mlp:g} µs (6 phép × 15 µs)")
ax.loglog(t_hw, t_sw_gpt3 / t_hw, color=MAU[1], lw=2, ls="--", label=f"phụ trội {t_sw_gpt3:g} µs")
ax.axhline(1, color="0.5", lw=1)
ax.fill_between(t_hw, 1, 1e4, color="0.92", zorder=0)
ax.annotate("tỉ số > 1: nghẽn ở phụ trội", (0.15, 2.0), fontsize=8, color="0.35")
ax.plot(t_hw_mlp, t_sw_mlp / t_hw_mlp, "o", color=MAU[0], ms=8)
ax.annotate(f"MLP nhỏ: {t_sw_mlp / t_hw_mlp:.1f}", (t_hw_mlp, t_sw_mlp / t_hw_mlp), xytext=(10, 4),
            textcoords="offset points", fontsize=9)
ax.plot(t_hw_gpt3, t_sw_gpt3 / t_hw_gpt3, "o", color=MAU[1], ms=8)
ax.annotate(f"tầng GPT-3: {t_sw_gpt3 / t_hw_gpt3:.4f}", (t_hw_gpt3, t_sw_gpt3 / t_hw_gpt3), xytext=(-60, -18),
            textcoords="offset points", fontsize=9)
ax.set_ylim(1e-5, 1e4)
ax.set_xlabel("thời gian phần cứng T_compute + T_memory (µs, thang log)")
ax.set_ylabel("tỉ số phụ trội (thang log)")
ax.set_title("Phụ trội điều phối chỉ đáng kể khi phép toán nhỏ")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Hai kịch bản cách nhau gần tám bậc độ lớn trên trục tung, dù khoản phụ trội gần như bằng nhau. Hệ
quả mà chương rút ra: số tham số một mình không quyết định mức lợi của việc biên dịch. Một tải công
việc (workload) gồm nhiều phép nhỏ có thể hưởng lợi không tương xứng khi việc điều phối chi phối,
vì hợp nhất làm giảm số lần điều phối. Sổ tay 02 đo đúng điều đó.

## 5. Một bước huấn luyện, đếm tới cùng

Chương lần theo một bước huấn luyện của perceptron hai tầng: batch 32, đầu vào 784, tầng ẩn 256,
10 lớp. Bảng 16 của chương tách công việc, lưu lượng bộ nhớ và mật độ tính toán (arithmetic
intensity) của từng phép. Ta dựng lại cả bảng từ hình dạng của các tensor, với các giả định chính
chương ghi: FP32, mỗi phép đọc đầu vào và ghi đầu ra đúng một lần, hàm mất mát tính ba phép cho mỗi
logit (mũ, cộng dồn, chuẩn hoá), lượt truyền ngược gấp đôi việc và gấp ba lưu lượng của lượt xuôi.
Sách đếm 1 KB là 1,000 byte.

In [ ]:
B_, D_, H_ = (sach(32, nguon="32 × 784 × 256"), sach(784, nguon="32 × 784 × 256"),
              sach(256, nguon="32 × 784 × 256"))
K_ = sach(10, trich="2 \\times 32 \\times 256{\\times}10")
BYTE = sach(4, nguon="4 byte → 1 byte")                # FP32
HE_SO_VIEC_NGUOC = sach(2, trich="Backward (assumed 2\\(\\times\\) forward)")
HE_SO_BYTE_NGUOC = 3   # chú thích bảng 16: "three times its memory traffic"
PHEP_MOI_LOGIT = len(["mũ", "cộng dồn", "chuẩn hoá"])  # chú thích bảng 16, ba phép cho mỗi logit

xuoi = {  # phép: (FLOP, byte đọc + ghi)
    "MatMul x @ W1": (2 * B_ * D_ * H_, (B_ * D_ + D_ * H_ + B_ * H_) * BYTE),
    "ReLU": (B_ * H_, 2 * B_ * H_ * BYTE),
    "MatMul h @ W2": (2 * B_ * H_ * K_, (B_ * H_ + H_ * K_ + B_ * K_) * BYTE),
    "entropy chéo": (PHEP_MOI_LOGIT * B_ * K_, 2 * B_ * K_ * BYTE),
}
flop_xuoi = sum(f for f, _ in xuoi.values())
byte_xuoi = sum(m for _, m in xuoi.values())
bang = dict(xuoi)
bang["lượt ngược"] = (HE_SO_VIEC_NGUOC * flop_xuoi, HE_SO_BYTE_NGUOC * byte_xuoi)

df = pd.DataFrame([(ten, f, m, f / m) for ten, (f, m) in bang.items()],
                  columns=["phép", "FLOP", "byte", "FLOP/byte"]).set_index("phép")
print(df.to_string(formatters={"FLOP": "{:,.0f}".format, "byte": "{:,.0f}".format, "FLOP/byte": "{:.3f}".format}))

theo_sach("việc, MatMul thứ nhất (MFLOP)", bang["MatMul x @ W1"][0] / 1e6, sach=12.8, trich="= 12.8 MFLOP")
theo_sach("việc, ReLU (KFLOP)", bang["ReLU"][0] / 1e3, sach=8.2, trich="= 8.2 KFLOP")
theo_sach("việc, MatMul thứ hai (KFLOP)", bang["MatMul h @ W2"][0] / 1e3, sach=163.8, trich="= 163.8 KFLOP")
theo_sach("việc, entropy chéo (KFLOP)", bang["entropy chéo"][0] / 1e3, sach=0.96, trich="~0.96 KFLOP", tol=0.005)
theo_sach("việc, lượt ngược (MFLOP)", bang["lượt ngược"][0] / 1e6, sach=26, trich="~26 MFLOP", tol=0.5)
theo_sach("lưu lượng, MatMul thứ nhất (MB)", bang["MatMul x @ W1"][1] / 1e6, sach=0.9, nguon="0,9 MB · 65,5 KB")
theo_sach("lưu lượng, ReLU (KB)", bang["ReLU"][1] / 1e3, sach=65.5, nguon="0,9 MB · 65,5 KB")
theo_sach("lưu lượng, MatMul thứ hai (KB)", bang["MatMul h @ W2"][1] / 1e3, sach=44.3, nguon="44,3 KB · 2,6 KB")
theo_sach("lưu lượng, entropy chéo (KB)", bang["entropy chéo"][1] / 1e3, sach=2.6, nguon="44,3 KB · 2,6 KB")
theo_sach("lưu lượng, lượt ngược (MB)", bang["lượt ngược"][1] / 1e6, sach=3.1, nguon="2,6 KB · 3,1 MB")
theo_sach("mật độ, MatMul thứ nhất", df.loc["MatMul x @ W1", "FLOP/byte"], sach=13.7, nguon="13,7 · 0,125 · 3,7")
theo_sach("mật độ, ReLU", df.loc["ReLU", "FLOP/byte"], sach=0.125, nguon="13,7 · 0,125 · 3,7")
theo_sach("mật độ, MatMul thứ hai", df.loc["MatMul h @ W2", "FLOP/byte"], sach=3.7, nguon="13,7 · 0,125 · 3,7")
theo_sach("mật độ, entropy chéo", df.loc["entropy chéo", "FLOP/byte"], sach=0.4, nguon="3,7 · 0,4 · 8,3 FLOP/byte")
theo_sach("mật độ, lượt ngược", df.loc["lượt ngược", "FLOP/byte"], sach=8.3, nguon="3,7 · 0,4 · 8,3 FLOP/byte")

Cột mật độ là cột dùng để chẩn đoán. Ta đặt nó cạnh điểm gãy của biểu đồ roofline (roofline) trên
A100: thông lượng đỉnh FP32 chia cho băng thông (bandwidth) bộ nhớ, cả hai lấy từ chương. Một phép
có mật độ thấp hơn điểm gãy thì nghẽn ở bộ nhớ (memory-bound) dù phần cứng còn dư năng lực tính.

In [ ]:
R_PEAK = sach(19.5, nguon="19,5 TFLOP/s · 2,0 µs") * 1e12   # FLOP/s, FP32 trên A100
BW_HBM = sach(2.04, nguon="2,04 TB/s · 2,1 µs") * 1e12      # byte/s
N_NAP = sach(12, nguon="12 lần nạp × 15")
flop_tong = flop_xuoi + bang["lượt ngược"][0]
byte_tong = byte_xuoi + bang["lượt ngược"][1]
T_compute = flop_tong / R_PEAK * 1e6   # µs
T_memory = byte_tong / BW_HBM * 1e6
T_phu_troi = N_NAP * T_DP
theo_sach("tổng việc (MFLOP)", flop_tong / 1e6, sach=39.1, nguon="39,1 MFLOP · 4,2 MB")
theo_sach("tổng lưu lượng (MB)", byte_tong / 1e6, sach=4.2, nguon="39,1 MFLOP · 4,2 MB")
theo_sach("chặn dưới theo tính toán (µs)", T_compute, sach=2.0, nguon="19,5 TFLOP/s · 2,0 µs")
theo_sach("chặn dưới theo bộ nhớ (µs)", T_memory, sach=2.1, nguon="2,04 TB/s · 2,1 µs")
theo_sach("phụ trội (µs)", T_phu_troi, sach=180, nguon="15 µs = 180 µs")
print(f"tỉ số phụ trội của bước này trên A100: {ti_so_phu_troi(N_NAP, T_DP, T_compute + T_memory):.0f}")

diem_gay = R_PEAK / BW_HBM
fig, ax = plt.subplots(figsize=(7, 3.8))
ten_phep = list(df.index)[::-1]
mat_do = [df.loc[t, "FLOP/byte"] for t in ten_phep]
ax.barh(ten_phep, mat_do, color=[MAU[0] if m > diem_gay else MAU[1] for m in mat_do], height=0.55)
for k, m in enumerate(mat_do):
    ax.annotate(f"{m:.3g}", (m, k), xytext=(4, 0), textcoords="offset points", va="center", fontsize=8)
ax.axvline(diem_gay, color="0.4", ls="--", lw=1)
ax.annotate(f"điểm gãy A100 FP32\n{diem_gay:.1f} FLOP/byte", (diem_gay, 1.5), xytext=(6, 0),
            textcoords="offset points", fontsize=8, va="center")
ax.set_xscale("log")
ax.set_xlim(0.05, 60)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=MAU[0], label="trên điểm gãy: nghẽn ở tính toán"),
                   Patch(color=MAU[1], label="dưới điểm gãy: nghẽn ở bộ nhớ")],
          loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=8, frameon=False)
ax.set_xlabel("mật độ tính toán (FLOP/byte, thang log)")
ax.set_title("Bảng 16 của chương: chỉ một phép vượt điểm gãy")
plt.tight_layout()
plt.show()

Việc thật của cả bước chỉ khoảng 2 µs nếu bị chặn bởi tính toán và khoảng 2 µs nếu bị chặn bởi bộ
nhớ. Với giả định 12 lần nạp, mỗi lần 15 µs, phụ trội là 180 µs, tức gấp hàng chục lần cả hai. Bước
huấn luyện được mô hình hoá này nghẽn ở phụ trội.

### Bước đó thật sự điều phối bao nhiêu phép?

Sách giả định 12 lần nạp kernel. Ta viết đúng bước ấy bằng PyTorch và đếm mọi phép mà bộ điều phối
nhận, kể cả các phép của lượt truyền ngược mà autograd sinh ra. `TorchDispatchMode` đứng ngay tại
chỗ PyTorch chọn kernel, nên nó thấy từng phép một. Có những phép chỉ đổi siêu dữ liệu tensor
(`t`, `view`, `detach`) mà không đọc hay ghi dữ liệu; ta đếm riêng chúng.

**Dự đoán:** nhiều hơn hay ít hơn 12?

In [ ]:
class DemPhep(TorchDispatchMode):
    """Ghi tên mọi phép aten đi qua bộ điều phối."""

    def __init__(self):
        super().__init__()
        self.ten = []

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        self.ten.append(func.overloadpacket.__name__)
        return func(*args, **(kwargs or {}))


CHI_SIEU_DU_LIEU = {"t", "view", "detach", "transpose", "expand", "as_strided", "_unsafe_view", "alias",
                    "permute", "reshape", "unsqueeze", "squeeze"}


def lam_buoc(b, h, lr=0.01):
    """Một bước huấn luyện của MLP 784-h-10 với batch b, viết như listing của chương."""
    dat_hat_giong()
    x, y = torch.randn(b, D_), torch.randint(0, K_, (b,))
    W1 = (0.05 * torch.randn(D_, h)).requires_grad_()
    b1 = torch.zeros(h, requires_grad=True)
    W2 = (0.05 * torch.randn(h, K_)).requires_grad_()
    b2 = torch.zeros(K_, requires_grad=True)

    def buoc():
        hh = torch.relu(x @ W1 + b1)
        logits = hh @ W2 + b2
        loss = F.cross_entropy(logits, y)
        loss.backward()
        with torch.no_grad():
            for p in (W1, b1, W2, b2):
                p -= lr * p.grad
                p.grad = None
    return buoc


buoc = lam_buoc(B_, H_)
buoc()
with DemPhep() as dem:
    buoc()
tinh_toan = [t for t in dem.ten if t not in CHI_SIEU_DU_LIEU]
print(f"tổng số phép đi qua bộ điều phối: {len(dem.ten)}")
print(f"  trong đó chỉ đổi siêu dữ liệu: {len(dem.ten) - len(tinh_toan)}")
print(f"  phép làm tính toán: {len(tinh_toan)}")
print("  theo tên:", dict(Counter(tinh_toan).most_common()))

Số phép tính toán mà PyTorch thật sự điều phối không phải 12: nó gồm cả lượt truyền ngược của từng
phép, phép tạo gradient khởi đầu và bốn lần cập nhật tham số. Con số chính xác đổi theo cách viết
bước huấn luyện và theo phiên bản PyTorch. Nếu mỗi phép tính toán là một lần nạp kernel trên GPU
thì phụ trội của bước này còn lớn hơn mô hình của sách.

### Cùng bước ấy trên CPU, khi tầng ẩn rộng dần

Trên A100, phần việc của bước chỉ vài micro giây nên phụ trội thắng. CPU tính chậm hơn nhiều, nên
chưa chắc như thế. Ô dưới đo thời gian một bước khi tầng ẩn rộng từ 1 tới 2048, cạnh mô hình A100
của chương (12 lần nạp × 15 µs, cộng hai chặn dưới) cho cùng các độ rộng.

**Dự đoán:** trên máy này, từ độ rộng nào trở đi thời gian một bước bắt đầu tăng theo số FLOP?

In [ ]:
cac_h = [1, 4, 16, 64, 256, 1024, 2048]
flop_h, t_cpu, t_a100 = [], [], []
for h in cac_h:
    f_xuoi = 2 * B_ * D_ * h + B_ * h + 2 * B_ * h * K_ + PHEP_MOI_LOGIT * B_ * K_
    m_xuoi = (B_ * D_ + D_ * h + B_ * h) * BYTE + 2 * B_ * h * BYTE + (B_ * h + h * K_ + B_ * K_) * BYTE + 2 * B_ * K_ * BYTE
    f = (1 + HE_SO_VIEC_NGUOC) * f_xuoi
    m = (1 + HE_SO_BYTE_NGUOC) * m_xuoi
    flop_h.append(f)
    t_a100.append(N_NAP * T_DP + f / R_PEAK * 1e6 + m / BW_HBM * 1e6)
    t = thoi_gian(lam_buoc(B_, h), lan=10)
    t_cpu.append(t * 1e6)
    do_tren_may(f"tầng ẩn {h:>4}: một bước trên CPU", t * 1e6, "µs")

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(flop_h, t_cpu, "o-", color=MAU[0], ms=7, label="CPU này, đo được")
ax.loglog(flop_h, t_a100, "s--", color=MAU[1], ms=6, label="A100, mô hình của chương")
ax.annotate("CPU: sàn phụ trội", (flop_h[0], t_cpu[0]), xytext=(0, 10), textcoords="offset points",
            fontsize=8, color="0.3")
ax.annotate("A100: 12 × 15 µs", (flop_h[0], t_a100[0]), xytext=(0, -16), textcoords="offset points",
            fontsize=8, color="0.3")
k256 = cac_h.index(H_)
ax.annotate("bảng 16 (tầng ẩn 256)", (flop_h[k256], t_a100[k256]), xytext=(12, -22), textcoords="offset points",
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_ylim(min(t_cpu[0], t_a100[0]) / 1.8, max(t_cpu) * 1.6)
moc = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]
ax.set_yticks([m for m in moc if ax.get_ylim()[0] <= m <= ax.get_ylim()[1]],
              [f"{m:,}" for m in moc if ax.get_ylim()[0] <= m <= ax.get_ylim()[1]])
ax.minorticks_off()
ax.set_xlabel("FLOP của một bước (xuôi + ngược, thang log)")
ax.set_ylabel("thời gian một bước (µs, thang log)")
ax.set_title("Cùng một bước huấn luyện, hai loại phần cứng")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Hai đường có cùng hình dạng: một sàn phẳng ở bên trái, nơi phụ trội chi phối, rồi một đoạn tăng
theo số FLOP ở bên phải. Khác nhau là chỗ gãy. A100 tính nhanh hơn CPU rất nhiều, nên đường của nó
còn nằm trên sàn phụ trội ở những độ rộng mà CPU đã chuyển sang nghẽn ở tính toán. Cùng một dòng mã,
cùng một bước huấn luyện, nhưng số hạng nào của định luật sắt (iron law) gánh thì tuỳ phần cứng bên
dưới. Ba cách chương nêu để giảm số hạng phụ trội đều nhắm vào sàn đó: biên dịch để hợp nhất và giảm
số lần nạp, tăng batch để phân bổ phụ trội cho nhiều mẫu hơn, và dùng phép lớn, đặc về tính toán.

## Thử thêm

1. Ở phần 3, thay `torch.add(x_, y_, out=z_)` bằng `torch.sin(x_, out=z_)`, một phép đắt hơn mỗi
   phần tử. Tung độ gốc $W$ đổi thế nào? Độ dốc $t_{dp}$ có đổi không? Điểm giao dịch sang trái hay
   sang phải?
2. Ở ô `tax_model`, đặt `T_DP` bằng phụ trội vừa đo trên máy này (`t_dp_may * 1e6`) rồi chạy lại phần
   5. Bước huấn luyện trên A100 có còn nghẽn ở phụ trội không? (Các dấu ✗ khi đó là điều được chờ đợi.)
3. Ở phần 5, tăng batch `B_` từ 32 lên 256. Theo mô hình của chương, tỉ số phụ trội trên A100 giảm
   khoảng bao nhiêu lần, và vì sao gần bằng 8?

## Tóm lại

* Mỗi lần điều phối trả một khoản cố định, bất kể phép toán lớn hay nhỏ. Thuế điều phối là phần
  thời gian nằm ở khoản đó, và nó chỉ đáng kể khi kernel ngắn.
* Đo trên máy này: chia cùng một lượng việc thành nhiều lần gọi hơn chỉ thêm phụ trội, và thời gian
  khớp với $n \cdot t_{dp} + W$.
* Định luật phụ trội điều phối so $N_{ops} \cdot t_{dp}$ với thời gian phần cứng. Tỉ số đó là 34.6
  cho một MLP nhỏ và 0.0005 cho một tầng GPT-3, dù khoản phụ trội gần như bằng nhau.
* Bước huấn luyện của bảng 16 có khoảng 2 µs việc thật và 180 µs phụ trội trên A100. Cùng bước đó
  trên CPU có thể nghẽn ở tính toán: số hạng nào gánh tuỳ phần cứng.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Framework — tầng giữa toán học và phần cứng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch07-frameworks/), dựng từ chương
[*ML Frameworks*](https://mlsysbook.ai/vol1/frameworks/frameworks.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_07_ml_frameworks.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.